In [ ]:
# ===== 1. 导入所需库 =====
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from transformers import Sam3Model, Sam3Processor
from quadrilateral_fitter import QuadrilateralFitter
from scipy.spatial import ConvexHull
from modelscope import snapshot_download
import tqdm

# 下载并加载 SAM3 模型
model_dir = snapshot_download('facebook/sam3', cache_dir='./sam3_model')
device = "cuda" if torch.cuda.is_available() else "cpu"
model_dir = "./sam3_model/facebook/sam3"

model = Sam3Model.from_pretrained(model_dir).to(device)
processor = Sam3Processor.from_pretrained(model_dir)

print(f"设备: {device}")


In [ ]:
# ===== 工具函数 =====
def _points_inside_quad(points, quad):
    """叉积法判断点是否在凸四边形内部"""
    pts, q = np.asarray(points, dtype=np.float64), np.asarray(quad, dtype=np.float64)
    signed_area = sum(q[i,0]*q[(i+1)%4,1] - q[(i+1)%4,0]*q[i,1] for i in range(4))
    is_cw = signed_area > 0
    inside = np.ones(len(pts), dtype=bool)
    for i in range(4):
        p1, p2 = q[i], q[(i+1)%4]
        dx, dy = p2[0]-p1[0], p2[1]-p1[1]
        cross = dx*(pts[:,1]-p1[1]) - dy*(pts[:,0]-p1[0])
        inside &= (cross >= 0) if is_cw else (cross <= 0)
    return inside

def _shrink_quad(vertices, factor=0.03):
    """沿对角线收缩四边形"""
    c = np.asarray(vertices, dtype=np.float64).mean(axis=0)
    return np.asarray(vertices) + factor * (c - vertices)

def _rectify_quality(vertices):
    """评估矫正质量"""
    w_top = np.linalg.norm(vertices[1]-vertices[0])
    w_bot = np.linalg.norm(vertices[2]-vertices[3])
    h_left = np.linalg.norm(vertices[3]-vertices[0])
    h_right = np.linalg.norm(vertices[2]-vertices[1])
    w_r = min(w_top,w_bot)/max(w_top,w_bot)
    h_r = min(h_left,h_right)/max(h_left,h_right)
    ratio = ((h_left+h_right)/2)/((w_top+w_bot)/2)
    a_s = min(ratio,1.35)/max(ratio,1.35)
    return w_r * h_r * a_s


In [ ]:
# 加载图片
image = Image.open("IMG_7131.jpg").convert("RGB")
image = ImageOps.exif_transpose(image)

plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.title('Input Image')
plt.axis('off')
plt.show()


In [ ]:
# ===== 3a. SAM3 分割拍立得纸框 =====
prompt = "polaroid photo paper frame"
inputs = processor(images=image, text=prompt, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model(**inputs)

results = processor.post_process_instance_segmentation(
    outputs, threshold=0.4, mask_threshold=0.5, target_sizes=[image.size[::-1]]
)[0]
paper_mask = results["masks"][0].cpu().numpy()

# ===== 3b. 轮廓提取 =====
contours, _ = cv2.findContours(
    paper_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE
)
points = np.vstack([c.reshape(-1, 2) for c in contours])

# ===== 3c. 四边形拟合（排除内部点） =====
hull = ConvexHull(points)
hull_pts = points[hull.vertices]

# 迭代最远点法 → 近似角点
p1 = hull_pts[np.argmin(hull_pts[:, 0])]
p2 = hull_pts[np.argmax(np.sum((hull_pts - p1) ** 2, axis=1))]
v = p2 - p1
signed = (v[0] * (hull_pts[:, 1] - p1[1]) - v[1] * (hull_pts[:, 0] - p1[0])) / np.linalg.norm(v)
approx = np.array([p1, p2, hull_pts[np.argmax(signed)], hull_pts[np.argmin(signed)]])
# 按绕中心的极角排序，确保四边形顶点顺序正确（避免蝴蝶形自交）
approx = approx[np.argsort(np.arctan2(
    approx[:, 1] - approx.mean(axis=0)[1],
    approx[:, 0] - approx.mean(axis=0)[0]
))]

# 收缩 + 排除内部点
center = approx.mean(axis=0)
shrunk = approx + 0.03 * (center - approx)
inside = _points_inside_quad(points, shrunk)  # 使用下方定义的工具函数
exterior = points[~inside]

# QuadrilateralFitter
fitter = QuadrilateralFitter(polygon=exterior)
paper_vertices = np.array(fitter.fit(), dtype=np.float64)
paper_vertices = paper_vertices[np.argsort(np.arctan2(
    paper_vertices[:, 1] - paper_vertices.mean(axis=0)[1],
    paper_vertices[:, 0] - paper_vertices.mean(axis=0)[0]
))]

# ===== 3d. 透视矫正 =====
src = paper_vertices.astype(np.float32)
w, h = 800, 1272  # 宽高比 1:1.59
dst = np.array([[0, 0], [w, 0], [w, h], [0, h]], dtype=np.float32)
M = cv2.getPerspectiveTransform(src, dst)
rectified_paper = cv2.warpPerspective(np.array(image), M, (w, h))
# 同步 warp 一个全1掩码 → 0=超出源图边界的缺失像素
valid_mask = cv2.warpPerspective(
    np.ones(np.array(image).shape[:2], dtype=np.uint8), M, (w, h)
).astype(bool)

# ===== 3e. 可视化 =====
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image); axes[0].set_title('Mask'); axes[0].axis('off')
axes[0].imshow(paper_mask, cmap='jet', alpha=0.5)

axes[1].imshow(image); axes[1].set_title('Quadrilateral'); axes[1].axis('off')
x = list(paper_vertices[:, 0]) + [paper_vertices[0, 0]]
y = list(paper_vertices[:, 1]) + [paper_vertices[0, 1]]
axes[1].plot(x, y, 'r-', lw=2)

axes[2].imshow(rectified_paper); axes[2].set_title('Rectified'); axes[2].axis('off')
plt.tight_layout(); plt.show()

print(f"纸框顶点: {paper_vertices.astype(int)}")
print(f"矫正尺寸: {rectified_paper.shape}")


In [ ]:
print(f"    纸框顶点: [{' '.join(str(row) for row in paper_vertices.astype(int))}]")

In [ ]:
# ===== 快捷入口：直接读取已矫正图片 =====
# 跳过 Cell 3（纸框检测+矫正），直接加载一张已矫正的拍立得图片
# 运行此 Cell 后接着运行 Cell 4（图像区域检测）即可继续流水线

import cv2
rectified_paper = cv2.cvtColor(cv2.imread("IMG_7066_001.png"), cv2.COLOR_BGR2RGB)
rectified_pil = Image.fromarray(rectified_paper)

print(f"已加载矫正图片: {rectified_paper.shape[1]}×{rectified_paper.shape[0]}")

plt.figure(figsize=(6, 8))
plt.imshow(rectified_paper)
plt.title('Loaded Rectified Image')
plt.axis('off')
plt.show()

In [ ]:
# ===== 4a. SAM3 分割图像区域 =====
rectified_pil = Image.fromarray(rectified_paper)
prompt = "the image area of the polaroid photo"
inputs = processor(images=rectified_pil, text=prompt, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model(**inputs)

results = processor.post_process_instance_segmentation(
    outputs, threshold=0.4, mask_threshold=0.5, target_sizes=[rectified_pil.size[::-1]]
)[0]
area_mask = results["masks"][0].cpu().numpy()

# ===== 4b. 轮廓 → 四边形拟合 =====
contours, _ = cv2.findContours(
    area_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE
)
points = np.vstack([c.reshape(-1, 2) for c in contours])

# ConvexHull + 近似 + 排除内部 → QuadrilateralFitter
hull = ConvexHull(points)
hull_pts = points[hull.vertices]
p1 = hull_pts[np.argmin(hull_pts[:, 0])]
p2 = hull_pts[np.argmax(np.sum((hull_pts - p1)**2, axis=1))]
v = p2 - p1
signed = (v[0]*(hull_pts[:,1]-p1[1]) - v[1]*(hull_pts[:,0]-p1[0])) / np.linalg.norm(v)
approx = np.array([p1, p2, hull_pts[np.argmax(signed)], hull_pts[np.argmin(signed)]])
# 按绕中心的极角排序，确保四边形顶点顺序正确（避免蝴蝶形自交）
approx = approx[np.argsort(np.arctan2(
    approx[:, 1] - approx.mean(axis=0)[1],
    approx[:, 0] - approx.mean(axis=0)[0]
))]
shrunk = _shrink_quad(approx, 0.03)
inside = _points_inside_quad(points, shrunk)
exterior = points[~inside]

fitter = QuadrilateralFitter(polygon=exterior)
area_vertices = np.array(fitter.fit(), dtype=np.float64)
area_vertices = area_vertices[np.argsort(np.arctan2(
    area_vertices[:,1]-area_vertices.mean(axis=0)[1],
    area_vertices[:,0]-area_vertices.mean(axis=0)[0]
))]

# ===== 4c. 透视矫正 + 质量 =====
# 固定尺寸拍立得 (800×1272) 的图像区域期望位置
# 标准拍立得上边距小、下边距大（底部留白写字）
EXPECTED_AREA_VERTICES = np.array([
    [55, 100],   # 左上
    [745, 100],  # 右上
    [745, 1022], # 右下
    [55, 1022],  # 左下
], dtype=np.float64)

# 角点距离误差（像素）
corner_error = np.mean(np.linalg.norm(area_vertices - EXPECTED_AREA_VERTICES, axis=1))

# 质量评分（基于长宽比 + 平行度）
quality = _rectify_quality(area_vertices)

# 综合：角点误差越小、quality 越高越好
print(f"角点平均偏差: {corner_error:.1f} px | 几何质量: {quality:.3f}")

# 透视矫正
src = area_vertices.astype(np.float32)
w, h = 685, 925  # 1:1.35
dst = np.array([[0,0],[w,0],[w,h],[0,h]], dtype=np.float32)
M = cv2.getPerspectiveTransform(src, dst)
rectified_area = cv2.warpPerspective(rectified_paper, M, (w, h))
quality = _rectify_quality(area_vertices)

# ===== 4d. 可视化：三张子图（凸包+近似 / 内外点划分 / 最终四边形） =====
pts = np.asarray(points, dtype=np.float64)
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 子图 1: 凸包 + 近似角点
axes[0].imshow(rectified_paper)
axes[0].scatter(hull_pts[:, 0], hull_pts[:, 1], c='cyan', s=5, alpha=0.6, label='Convex Hull')
axes[0].scatter(approx[:, 0], approx[:, 1], c='red', s=50, zorder=5, label='Approx Corners')
axes[0].set_title('1. Convex Hull & Approx Corners')
axes[0].legend(loc='upper right', fontsize=8)
axes[0].axis('off')

# 子图 2: 内部/外部点划分
exterior_sample = exterior if len(exterior) <= 5000 else exterior[np.random.choice(len(exterior), 5000, replace=False)]
interior_pts = pts[inside]
interior_sample = interior_pts if len(interior_pts) <= 5000 else interior_pts[np.random.choice(len(interior_pts), 5000, replace=False)]

axes[1].imshow(rectified_paper)
# 收缩四边形虚线
x = list(shrunk[:, 0]) + [shrunk[0, 0]]
y = list(shrunk[:, 1]) + [shrunk[0, 1]]
axes[1].plot(x, y, 'y--', linewidth=1.5, alpha=0.8, label='Shrunk Quad')
axes[1].scatter(exterior_sample[:, 0], exterior_sample[:, 1], c='green', s=1, alpha=0.4, label=f'Exterior ({len(exterior)})')
axes[1].scatter(interior_sample[:, 0], interior_sample[:, 1], c='orange', s=1, alpha=0.4, label=f'Interior ({np.sum(inside)})')
axes[1].set_title('2. Interior / Exterior Split')
axes[1].legend(loc='upper right', fontsize=8)
axes[1].axis('off')

# 子图 3: 最终四边形
vertices_int = area_vertices.astype(int)
axes[2].imshow(rectified_paper)
x = list(vertices_int[:, 0]) + [vertices_int[0, 0]]
y = list(vertices_int[:, 1]) + [vertices_int[0, 1]]
axes[2].plot(x, y, 'r-', linewidth=2.5, label='Fitted Quadrilateral')
axes[2].scatter(vertices_int[:, 0], vertices_int[:, 1], c='red', s=60, zorder=5)
axes[2].set_title(f'3. Final Quadrilateral (q={quality:.3f})')
axes[2].legend(loc='upper right', fontsize=8)
axes[2].axis('off')
plt.tight_layout(); plt.show()
print(f"图像区域顶点: {area_vertices.astype(int)}, 质量: {quality:.3f}")


In [ ]:
# ===== 5a. 生成边框掩码 =====
h, w = rectified_paper.shape[:2]
border_mask = np.ones((h, w), dtype=np.uint8)

margin = 5
inner = np.array([
    [area_vertices[0][0]-margin, area_vertices[0][1]-margin],
    [area_vertices[1][0]+margin, area_vertices[1][1]-margin],
    [area_vertices[2][0]+margin, area_vertices[2][1]+margin],
    [area_vertices[3][0]-margin, area_vertices[3][1]+margin]
], dtype=np.int32)
cv2.fillPoly(border_mask, [inner], 0)
border_mask = border_mask.astype(bool)

# ===== 5b. 寻找白色参考区域 =====
img_arr = np.array(rectified_pil)
is_bright = np.all(img_arr > 200, axis=2)
is_neutral = np.std(img_arr.astype(np.float32), axis=2) < 15
is_white = is_bright & is_neutral & border_mask

blocks = []
for y in range(0, h-32, 16):
    for x in range(0, w-32, 16):
        block = is_white[y:y+32, x:x+32]
        if np.sum(block) / 1024 > 0.8:
            pixels = img_arr[y:y+32, x:x+32][block]
            blocks.append({'x': x, 'y': y, 'mean': pixels.mean(axis=0), 'var': pixels.var(axis=0).mean()})

if blocks:
    blocks.sort(key=lambda b: b['var'])
    best = blocks[:10]
    ref_white = np.mean([b['mean'] for b in best], axis=0)
    target = 240.0  # 固定目标白色 (240, 240, 240)
    gains = np.array([target/ref_white[0], target/ref_white[1], target/ref_white[2]])
    wb_image = np.clip(img_arr.astype(np.float32) * gains, 0, 255).astype(np.uint8)
    print(f"白平衡增益: R={gains[0]:.3f} G={gains[1]:.3f} B={gains[2]:.3f} → 目标 (240,240,240)")
else:
    wb_image = img_arr.copy()
    best = []
    target = 240.0
    print("未找到白色参考区域，跳过白平衡")

# ===== 5c. 检测真正的缺失区域（warpPerspective 超出源图边界） =====
# 用 Cell 3 中同步 warp 的 valid_mask —— 0=源图范围外的缺失像素，精确不误伤
missing_mask = ~valid_mask

# 膨胀覆盖交界处黑线及暗边过渡
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
missing_mask = cv2.dilate(missing_mask.astype(np.uint8), kernel).astype(bool)

true_missing = missing_mask.sum()
print(f"真正缺失像素（warp 超出源图边界）: {true_missing}")

if missing_mask.any():
    wb_image[missing_mask] = (240, 240, 240)
    print(f"填充了 {true_missing} 个缺失像素 → (240, 240, 240)")

# 保存缺失区域掩码
cv2.imwrite("missing_region_mask.png", (missing_mask.astype(np.uint8) * 255))
print("缺失区域掩码已保存: missing_region_mask.png")

# ===== 5d. 可视化 =====
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(rectified_pil); axes[0].set_title('Before WB'); axes[0].axis('off')

# 白参考块（绿框）+ 缺失区域（红）
vis = np.array(rectified_pil).copy()
for b in best:
    cv2.rectangle(vis, (b['x'], b['y']), (b['x']+32, b['y']+32), (0, 255, 0), 2)
if missing_mask.any():
    vis[missing_mask] = (255, 0, 0)
axes[1].imshow(vis); axes[1].set_title(f'White Ref ({len(best)} blocks) + Missing'); axes[1].axis('off')

axes[2].imshow(wb_image); axes[2].set_title(f'After WB + Fill → (240,240,240)'); axes[2].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# ===== 6a. SAM3 多 Prompt × 多阈值网格搜索 =====
wb_pil = Image.fromarray(wb_image)
prompts = [
    "ink",
    "handwriting",
    "writing",
    "text",
    "number",
    "scribble",
    "black ink",
    "the ink marks on the photo",
    "pen writing on the image",
    "ballpoint pen handwriting on the photo",
    "handwritten signature and notes on polaroid",
    "hand-drawn marks and writing on the image",
    "the handwritten words and marks on this polaroid picture",
    "all ink, pen marks, and handwriting visible on the photo",
]
thresholds = [0.5, 0.4, 0.3, 0.25, 0.2, 0.15, 0.1, 0.05]

all_results = []

for prompt in tqdm.tqdm(prompts):
    inputs = processor(images=wb_pil, text=prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)

    for thresh in thresholds:
        results = processor.post_process_instance_segmentation(
            outputs, threshold=thresh, mask_threshold=0.5,
            target_sizes=[wb_pil.size[::-1]]
        )[0]

        num_masks = len(results["masks"])
        if num_masks == 0:
            all_results.append({
                'prompt': prompt, 'threshold': thresh,
                'num_masks': 0, 'has_large': False,
                'masks': None, 'ink_mask': None
            })
            continue

        masks = results["masks"].cpu().numpy()
        mask_areas = np.mean(masks, axis=(1, 2))
        has_large = np.any(mask_areas > 0.5)
        ink_mask = np.any(masks, axis=0)

        all_results.append({
            'prompt': prompt, 'threshold': thresh,
            'num_masks': num_masks, 'has_large': bool(has_large),
            'masks': masks, 'ink_mask': ink_mask
        })

# ===== 6b. 汇总输出 =====
print(f"{'Prompt':<55} {'Thresh':>6} {'Masks':>6} {'Large':>6}")
print("-" * 75)
for r in all_results:
    large_str = "⚠️" if r['has_large'] else ""
    print(f"{r['prompt']:<55} {r['threshold']:>6} {r['num_masks']:>6} {large_str:>6}")

# ===== 6c. 可视化网格 =====
n_prompts = len(prompts)
n_thresh = len(thresholds)
fig, axes = plt.subplots(n_prompts, n_thresh, figsize=(n_thresh * 2.2, n_prompts * 2.2))

for i, prompt in enumerate(prompts):
    for j, thresh in enumerate(thresholds):
        ax = axes[i, j]
        r = all_results[i * n_thresh + j]
        if r['ink_mask'] is not None:
            overlay = wb_image.copy()
            overlay[r['ink_mask']] = (255, 0, 0)
            ax.imshow(cv2.addWeighted(wb_image, 0.5, overlay, 0.5, 0))
        else:
            ax.imshow(wb_image)
        title = f"t={thresh} n={r['num_masks']}"
        if j == 0:
            title += f"\n{prompt}"
        if r['has_large']:
            title += " ⚠️"
        ax.set_title(title, fontsize=7)
        ax.axis('off')
        if j == 0:
            ax.set_ylabel(prompt, fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# ===== 释放 CUDA 显存 =====
import gc
del all_results, outputs, inputs
gc.collect()
torch.cuda.empty_cache()
print(f"CUDA 显存已释放，当前分配: {torch.cuda.memory_allocated() / 1024**2:.0f} MB / 缓存: {torch.cuda.memory_reserved() / 1024**2:.0f} MB")
